# transforms

> Albumentations augmentation pipelines that work for any channel count, plus per-channel normalization statistics.

The transforms operate on `(H, W, C)` images and integer-label masks together, so geometric augmentations stay aligned between image and mask. Normalization mean/std are per-channel sequences, so the same pipeline serves 2-, 3- or N-channel inputs.

In [ ]:
#| default_exp transforms

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Iterable, Mapping, Optional, Sequence, Tuple

import cv2
import numpy as np
import albumentations as A
from albumentations.core.transforms_interface import ImageOnlyTransform
from albumentations.pytorch import ToTensorV2
import torch


## Per-channel statistics

Before training we estimate the mean/std of each channel so `Normalize` centres the data. We compute it in `[0, 1]` scale (after dividing by 255), which is what albumentations expects when `max_pixel_value=255`.

In [ ]:
# experiment: per-channel mean/std over a few synthetic images
rng = np.random.default_rng(0)
imgs = [rng.integers(0, 256, (16, 16, 3), np.uint8) for _ in range(4)]
stacked = np.stack(imgs).astype(np.float32) / 255.0  # (N, H, W, C)
print('mean per channel:', stacked.mean(axis=(0, 1, 2)).round(3))
print('std  per channel:', stacked.std(axis=(0, 1, 2)).round(3))

In [ ]:
#| export
def compute_dataset_stats(
    images: Iterable[np.ndarray],  # iterable of (H, W, C) uint8 images
) -> Tuple[Tuple[float, ...], Tuple[float, ...]]:
    """Streaming per-channel `(mean, std)` in `[0, 1]` scale.

    Uses a running sum / sum-of-squares so it scales to large datasets without
    holding every image in memory.

    Example:
        >>> m, s = compute_dataset_stats([np.full((4, 4, 1), 128, np.uint8)])
        >>> round(m[0], 3)
        0.502
    """
    n = 0
    s = s2 = None
    for img in images:
        x = img.astype(np.float64).reshape(-1, img.shape[-1]) / 255.0
        if s is None:
            s = np.zeros(x.shape[1]); s2 = np.zeros(x.shape[1])
        n += x.shape[0]; s += x.sum(0); s2 += (x ** 2).sum(0)
    mean = s / n
    std = np.sqrt(np.maximum(s2 / n - mean ** 2, 1e-12))
    return tuple(mean.tolist()), tuple(std.tolist())

In [ ]:
# test
m, s = compute_dataset_stats([np.full((4, 4, 1), 128, np.uint8)])
assert abs(m[0] - 128 / 255) < 1e-3 and s[0] < 1e-3

## Mask-guided boundary blur

Simulates indistinct chip edges: blur and optional halo brightening in a band
around the mask contour. The mask stays sharp so the model learns geometry from
interior structure, not from a crisp intensity step.


In [ ]:
#| export
def _odd_kernel(radius: int) -> int:
    k = int(radius) | 1
    return max(k, 3)


def _binarize_mask_for_blur(mask: np.ndarray) -> np.ndarray:
    if mask.dtype == np.bool_:
        return mask.astype(np.uint8)
    if mask.max() <= 1:
        return (mask > 0).astype(np.uint8)
    return (mask > 127).astype(np.uint8)


class BlurBoundaryZone(ImageOnlyTransform):
    """Blur image pixels in a band around the mask edge; mask is unchanged."""

    def __init__(
        self,
        band_width: Sequence[int] = (5, 21),
        sigma: Sequence[float] = (3, 11),
        halo_brightness: Sequence[float] = (0.0, 0.15),
        p: float = 0.5,
    ):
        super().__init__(p=p)
        self.band_width = tuple(band_width)
        self.sigma = tuple(sigma)
        self.halo_brightness = tuple(halo_brightness)

    @property
    def targets_as_params(self):
        return ["mask"]

    def get_params_dependent_on_data(self, params, data):
        mask = data.get("mask")
        if mask is None:
            return {"alpha": None}
        m = _binarize_mask_for_blur(mask)
        if m.max() == 0:
            return {"alpha": None}
        lo, hi = int(self.band_width[0]), int(self.band_width[1])
        if lo > hi:
            lo, hi = hi, lo
        if lo == hi:
            radius = lo if lo % 2 == 1 else lo + 1
        else:
            radius = self.py_random.randrange(lo, hi + 1, 2)
        k = _odd_kernel(radius)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (k, k))
        chans = [m] if m.ndim == 2 else [np.ascontiguousarray(m[..., c]) for c in range(m.shape[-1])]  # multilabel (H,W,K)
        band = np.zeros(m.shape[:2], np.uint8)
        for ch in chans:  # every class edge gets blurred, incl. sub-class edges inside a super-class
            band |= ((cv2.dilate(ch, kernel) - cv2.erode(ch, kernel)) > 0).astype(np.uint8)
        if band.max() == 0:
            return {"alpha": None}
        dist = cv2.distanceTransform(band, cv2.DIST_L2, 3)
        alpha = np.clip(dist / max(float(dist.max()), 1e-6), 0, 1).astype(np.float32)
        sig_lo, sig_hi = float(self.sigma[0]), float(self.sigma[1])
        sigma = self.py_random.uniform(min(sig_lo, sig_hi), max(sig_lo, sig_hi))
        h_lo, h_hi = float(self.halo_brightness[0]), float(self.halo_brightness[1])
        halo = (
            self.py_random.uniform(min(h_lo, h_hi), max(h_lo, h_hi))
            if max(h_lo, h_hi) > 0
            else 0.0
        )
        fg = m if m.ndim == 2 else np.ascontiguousarray(m.max(axis=-1))  # union of classes
        outer = ((cv2.dilate(fg, kernel) - fg) > 0).astype(np.float32)
        return {"alpha": alpha, "sigma": sigma, "halo": halo, "outer": outer}

    def apply(self, img, alpha=None, sigma=5.0, halo=0.0, outer=None, **params):
        if alpha is None:
            return img
        img_f = img.astype(np.float32)
        ksize = int(max(3, round(sigma * 4))) | 1
        blurred = cv2.GaussianBlur(img_f, (ksize, ksize), sigmaX=sigma, sigmaY=sigma)
        if img_f.ndim == 3 and blurred.ndim == 2:
            blurred = blurred[..., None]  # cv2 drops the channel axis of (H, W, 1) images
        if halo > 0 and outer is not None:
            boost = halo * 255.0 * outer
            blurred = blurred + (boost[..., None] if img_f.ndim == 3 else boost)
        blurred = np.clip(blurred, 0, 255)
        if img_f.ndim == 2:
            out = img_f * (1 - alpha) + blurred * alpha
        else:
            out = img_f * (1 - alpha[..., None]) + blurred * alpha[..., None]
        return np.clip(out, 0, 255).astype(img.dtype)


def boundary_blur_kwargs_from_cfg(aug_cfg: Mapping) -> dict:
    """Extract boundary-blur settings from a Hydra/OmegaConf `augment` section."""
    p = float(aug_cfg.get("boundary_blur", 0) or 0)
    if p <= 0:
        return {}

    def _pair(key: str, default: tuple):
        v = aug_cfg.get(key, default)
        if v is None:
            return default
        return tuple(v)

    return dict(
        p=p,
        band_width=_pair("boundary_blur_width", (5, 21)),
        sigma=_pair("boundary_blur_sigma", (3, 11)),
        halo_brightness=_pair("boundary_halo_brightness", (0.0, 0.15)),
    )


In [ ]:
# test BlurBoundaryZone
_img = np.zeros((64, 64, 3), np.uint8)
_img[:, :] = 200
_img[20:40, 20:40] = 50
_msk = np.zeros((64, 64), np.uint8); _msk[20:40, 20:40] = 1
_t = BlurBoundaryZone(p=1.0)
_out = _t(image=_img, mask=_msk)
assert np.array_equal(_out["mask"], _msk)
assert not np.allclose(_out["image"][20, 30], _img[20, 30])
assert np.allclose(_out["image"][0, 0], _img[0, 0])
assert np.allclose(_out["image"][30, 30], _img[30, 30])
_empty = _t(image=_img, mask=np.zeros((64, 64), np.uint8))
assert np.array_equal(_empty["image"], _img)


Multilabel masks arrive as `(H, W, K)` (one channel per class). The band step used to assume one 2-D mask, so
`augment.boundary_blur > 0` crashed OpenCV on the multilabel recipe. The band is now the union of every class's edge,
so a sub-class edge inside its super-class (`lifted` inside `chip`) is blurred too.

In [ ]:
# test: BlurBoundaryZone accepts a multilabel (H, W, K) mask and blurs around the inner class edge too
_img = np.zeros((64, 64, 3), np.uint8); _img[:, :] = 200; _img[28:36, 28:36] = 0   # dark inner square
_m = np.zeros((64, 64, 2), np.uint8); _m[10:54, 10:54, 0] = 1; _m[26:38, 26:38, 1] = 1   # class 1 inside class 0
_t = BlurBoundaryZone(band_width=(5, 5), sigma=(4, 4), halo_brightness=(0.0, 0.0), p=1.0)
_out = _t(image=_img, mask=_m)['image']
assert _out.shape == _img.shape and not np.array_equal(_out, _img)
assert not np.array_equal(_out[24:40, 24:40], _img[24:40, 24:40])   # inner (class 1) edge region changed
print('BlurBoundaryZone ✓  multilabel (H,W,K) mask')

In [ ]:
# test: single-channel (H, W, 1) images keep their shape (cv2.GaussianBlur drops the channel axis)
_img1 = np.full((64, 64, 1), 200, np.uint8); _img1[28:36, 28:36] = 0
_t1 = BlurBoundaryZone(band_width=(5, 5), sigma=(4, 4), halo_brightness=(0.0, 0.1), p=1.0)
_o1 = _t1(image=_img1, mask=_m)['image']
assert _o1.shape == _img1.shape and not np.array_equal(_o1, _img1)
print('BlurBoundaryZone ✓  (H,W,1) image shape preserved')

## Train and validation pipelines

Train-time augmentation models realistic nuisance variation (placement jitter, brightness, mild blur/noise) without changing target geometry. Flips are **off by default**: many inspection layouts have a fixed orientation, so mirrored boards would be unrealistic. Validation only crops and normalizes.

In [ ]:
#| export
def get_train_transforms(
    crop_size: int = 256,                  # output square crop size
    mean: Sequence[float] = (0., 0., 0.),  # per-channel mean (length == channels)
    std: Sequence[float] = (1., 1., 1.),   # per-channel std (length == channels)
    aug=None,                              # reserved for future AugmentConfig support
    flips: bool = False,                   # enable h/v flips (only if layout is symmetric)
    boundary_blur: Optional[dict] = None,  # from boundary_blur_kwargs_from_cfg / Hydra
) -> A.Compose:
    """Augmentation pipeline for training. Channel-count agnostic via `mean`/`std`."""
    aug_list = [
        A.RandomCrop(height=crop_size, width=crop_size, p=1.0),
        A.Affine(translate_percent={"x": (-0.04, 0.04), "y": (-0.04, 0.04)},
                 scale=(0.92, 1.08), rotate=(-5, 5),
                 border_mode=cv2.BORDER_REFLECT_101, p=0.6),
        A.GridDistortion(num_steps=5, distort_limit=0.04, border_mode=cv2.BORDER_REFLECT_101, p=0.25),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.7),
        A.GaussNoise(std_range=(0.02, 0.10), p=0.5),
        A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    ]
    if boundary_blur:
        aug_list.append(BlurBoundaryZone(**boundary_blur))
    if flips:
        aug_list += [A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5)]
    aug_list += [A.Normalize(mean=tuple(mean), std=tuple(std), max_pixel_value=255.0), ToTensorV2()]
    return A.Compose(aug_list)

def get_val_transforms(
    crop_size: int = 256,                  # output square crop size
    mean: Sequence[float] = (0., 0., 0.),  # per-channel mean
    std: Sequence[float] = (1., 1., 1.),   # per-channel std
) -> A.Compose:
    """Deterministic centre-crop + normalize pipeline for validation."""
    return A.Compose([
        A.CenterCrop(height=crop_size, width=crop_size, p=1.0),
        A.Normalize(mean=tuple(mean), std=tuple(std), max_pixel_value=255.0),
        ToTensorV2(),
    ])


In [ ]:
# test: image -> (C, crop, crop) tensor; mask stays aligned. Works for 3 and 4 channels.
_bb = dict(p=1.0, band_width=(5, 5), sigma=(3, 3), halo_brightness=(0.0, 0.0))
for C in (3, 4):
    img = rng.integers(0, 256, (64, 64, C), np.uint8)
    msk = np.zeros((64, 64), np.uint8); msk[20:40, 20:40] = 1
    z = (0.,) * C; o = (1.,) * C
    out = get_train_transforms(32, z, o, boundary_blur=_bb)(image=img, mask=msk)
    assert out["image"].shape == (C, 32, 32)
    assert out["mask"].shape == (32, 32)
    vout = get_val_transforms(32, z, o)(image=img, mask=msk)
    assert vout["image"].shape == (C, 32, 32)


## Un-normalize for display

Reverse `Normalize` so we can show a normalized tensor as an image.

In [ ]:
#| export
def unnormalize(
    image: torch.Tensor,                   # (C, H, W) normalized tensor
    mean: Sequence[float] = (0., 0., 0.),  # per-channel mean used in Normalize
    std: Sequence[float] = (1., 1., 1.),   # per-channel std used in Normalize
) -> torch.Tensor:
    """Invert `Normalize`, returning a `(C, H, W)` tensor clamped to `[0, 1]`."""
    c = image.shape[0]
    m = torch.tensor(mean[:c]).view(c, 1, 1)
    s = torch.tensor(std[:c]).view(c, 1, 1)
    return (image * s + m).clamp(0, 1)

In [ ]:
# test
t = torch.rand(3, 8, 8)
out = unnormalize(t, (0., 0., 0.), (1., 1., 1.))
assert out.shape == (3, 8, 8) and float(out.max()) <= 1.0

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()